# BTC 48-hour magnitude — Random Forest, walk-forward evaluation & MLflow

**Target:** `net_magnitude_log_bps` • **Primary metric:** mean absolute error (MAE), in log basis points.

Run the cells in order in Google Colab. Your source Parquet is opened read-only. Outputs go to a new timestamped experiment folder in Drive.

## Design
- Candidate X: only the 522 `f4z__...__value` columns from your schema. `__valid` and family `__available_after` are used to mask values, not as raw predictors. Missing-value indicators are created inside each fit.
- Use the ZIP's initial admission: at least 1,000 finite observations and at least two distinct values **inside the current training fold**.
- Compare all admitted features, a standalone median-regression MAE screen, and a KS/lift screen. All screening is inside training data. The MAE screen uses its own purged chronological validation segment; if no single beats its training-median baseline there, it falls back to KS/lift. KS/lift is also a separately predeclared alternative, not a response to final test results.
- Tune using three earlier, monthly expanding folds. Freeze the winning selection strategy and forest hyperparameters **before** the last three months are evaluated.
- Evaluate the last three calendar months represented by valid labels, refitting once per month using only labels whose 48-hour endpoints precede that month's start. The latest month can be partial. Earlier evaluation months may enter later training once their labels mature: this is expanding walk-forward evaluation.
- Default sampling is every 30 minutes, as in the ZIP. No averaging or interpolation of rows.
- Compare against a freshly fitted training-median constant on exactly the same evaluation rows. Report pooled MAE, monthly MAE, and nominal paired 7/14-day block bootstrap intervals.

**Not an exact replay of the ZIP:** this uses your larger Parquet, your named target, a new date split, and monthly refitting. Do not compare its score directly with the ZIP's 182.036977 score on different rows. Confirm that `net_magnitude_log_bps` means absolute 48-hour endpoint log return, not a high/low excursion or cost-adjusted target. Finite nonnegative values alone cannot verify this definition.

This is a new experiment: the default search makes **36 tuning RF fits + 3 monthly RF fits**, plus standalone screening regressions. It exceeds the ZIP's original 20-fit/member budget; all attempts are logged. Reduce the search if that budget still applies.

The real Drive dataset was not available during notebook construction. Synthetic checks validate the loader, purging, screening, and evaluation helpers, not real-data performance. MLflow integration was not executed successfully in the authoring environment: automatic approval review blocked an outbound telemetry connection. Optional MLflow telemetry is disabled in this notebook before import.

## 1. Install dependencies
Use a fresh CPU runtime. Random forests do not use Colab's GPU. Absolute-error trees align better with the magnitude objective but can be slow; expect a substantial CPU run. These version ranges are recorded exactly with `pip freeze` for each experiment.

In [ ]:
%pip -q install "numpy>=1.26,<3" "pandas>=2.2,<3" "pyarrow>=17,<24" "scipy>=1.13,<2" "scikit-learn>=1.5,<2" "mlflow>=2.17,<4" "joblib>=1.4,<2" matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, hashlib, sqlite3, shutil, subprocess, warnings, time
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from scipy.stats import ks_2samp
from sklearn.linear_model import QuantileRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import mean_absolute_error
import joblib
# Disable optional outbound MLflow telemetry before importing MLflow.
os.environ['MLFLOW_DISABLE_TELEMETRY'] = 'true'
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
from IPython.display import display

file_path = '/content/drive/MyDrive/Scalewave/Datasets/X,Y Dataset/HANDOFF_2_CANONICAL_XY_V1_1/y/horizon_minutes=2880/symbol=BTCUSDT/combined_X_y_BTCUSDT_48hr_dataset_without_f4.parquet'
TARGET = 'net_magnitude_log_bps'
HORIZON = pd.Timedelta(hours=48)
CADENCE_MINUTES = 30
MIN_VALID = 1000
TOP_K = 20
SEED = 48
RF_N_JOBS = 2                 # avoid exhausting Colab RAM
MAX_X_RAM_MB = 750            # stop rather than silently exhaust RAM
BLEND_WEIGHTS = [0.0, 0.25, 0.5, 0.75, 1.0]  # 0=baseline, 1=pure RF
CONFIGS_PER_STRATEGY = 4      # 3 strategies * 4 configs * 3 folds = 36 RF fits
N_BOOTSTRAP = 2000

# Set these after inspecting the printed status counts in cell 3.
# None prompts you to type the exact accepted label_status value(s).
# If the column is absent, only finite/nonnegative target admission is possible.
VALID_LABEL_STATUSES = None

DROP_FROM_X = [
    'horizon_minutes', 'direction', 'up_magnitude_log_bps',
    'down_magnitude_log_bps', 'time_to_high_min', 'time_to_low_min',
    'extreme_order', 'label_status', 'symbol', 'source_month',
    'canonical_symbol', 'decision_time', TARGET,
]
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUTPUT_ROOT = Path('/content/drive/MyDrive/Scalewave/Experiments/RF_48h_magnitude') / RUN_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)
LOCAL_ROOT = Path('/content') / ('rf48_' + RUN_ID)
LOCAL_ROOT.mkdir(parents=True, exist_ok=False)
print('Outputs:', OUTPUT_ROOT)
print('Source file is never overwritten.')

## 2. Read and mask the data in batches
The loader selects only required columns, casts Arrow decimals to floating point, filters to exact UTC 30-minute origins, and masks invalid/late feature readings. It does not load thousands of reason/status columns into X. It rejects mixed symbols, incorrect horizons, and duplicate timestamps rather than silently aggregating them.

The label endpoint is derived as `decision_time + 48h`; if a `label_end` column exists, it must agree. A family-level `available_after` mask is conservative. It does not certify all upstream source construction or normalization, including the historical funding-boundary limitations noted in the ZIP.

In [ ]:
def load_masked_dataset(path, target, cadence=30):
    pf = pq.ParquetFile(path)
    names = pf.schema_arrow.names
    if len(names) != len(set(names)):
        raise ValueError('Duplicate column names in source.')
    values = sorted(c for c in names if c.lower().startswith('f4z__') and c.endswith('__value'))
    if not values or target not in names or 'decision_time' not in names:
        raise ValueError('Missing F4Z values, decision_time, or target.')
    validity = {c: c[:-len('__value')] + '__valid' for c in values}
    availability = {c: '__'.join(c.split('__')[:2]) + '__available_after' for c in values}
    required_masks = set(validity.values()) | set(availability.values())
    missing_masks = required_masks - set(names)
    if missing_masks:
        raise ValueError(f'Missing validity/availability fields: {sorted(missing_masks)[:10]}')
    metadata = [c for c in ['canonical_symbol', 'symbol', 'horizon_minutes', 'label_status', 'label_end'] if c in names]
    read_cols = ['decision_time', target] + metadata + values + sorted(required_masks)
    chunks, audits = [], []
    raw_rows = sampled_rows = 0
    for batch_no, batch in enumerate(pf.iter_batches(batch_size=4096, columns=read_cols)):
        raw_rows += batch.num_rows
        tab = pa.Table.from_batches([batch])
        dt = pd.DatetimeIndex(pd.to_datetime(tab['decision_time'].to_pandas(), utc=True))
        if dt.isna().any():
            raise ValueError('Missing decision_time.')
        mask = (dt.asi8 % pd.Timedelta(minutes=cadence).value) == 0
        if not mask.any():
            continue
        tab = tab.filter(pa.array(mask))
        # Avoid creating hundreds of Python Decimal objects per row.
        for col in values + [target]:
            idx = tab.schema.get_field_index(col)
            tab = tab.set_column(idx, col, pc.cast(tab[col], pa.float64(), safe=False))
        d = tab.to_pandas()
        d['decision_time'] = pd.to_datetime(d['decision_time'], utc=True)
        for col in ['canonical_symbol', 'symbol']:
            if col in d and not d[col].eq('BTCUSDT').all():
                raise ValueError(f'{col} contains missing/non-BTCUSDT rows.')
        if 'horizon_minutes' in d and not pd.to_numeric(d.horizon_minutes, errors='coerce').eq(2880).all():
            raise ValueError('Some rows do not have a 2880-minute horizon.')
        if 'label_end' in d:
            le = pd.to_datetime(d.label_end, utc=True)
            if not le.eq(d.decision_time + pd.Timedelta(hours=48)).all():
                raise ValueError('label_end does not equal decision_time + 48 hours.')
        arrays = {}
        for col in values:
            valid = d[validity[col]].eq(True).fillna(False).to_numpy(dtype=bool)
            avail = pd.to_datetime(d[availability[col]], utc=True, errors='coerce')
            on_time = (avail.notna() & (avail <= d.decision_time)).to_numpy()
            v = d[col].to_numpy(dtype=float)
            bad = ~(valid & on_time & np.isfinite(v))
            v[bad] = np.nan
            arrays[col] = v.astype(np.float32)
        out = pd.DataFrame(arrays, index=pd.DatetimeIndex(d.decision_time))
        out[target] = d[target].to_numpy(dtype=float)
        if 'label_status' in d:
            out['label_status'] = d.label_status.to_numpy()
        chunks.append(out)
        sampled_rows += len(out)
        estimated_mb = sampled_rows * len(values) * 4 / 1024**2
        if estimated_mb > MAX_X_RAM_MB:
            raise MemoryError(f'X exceeds {MAX_X_RAM_MB} MB budget. Increase budget on a larger runtime; do not silently drop rows.')
        if batch_no % 100 == 0:
            print(f'Read {raw_rows:,} raw rows; retained {sampled_rows:,} cadence rows', flush=True)
    if not chunks:
        raise ValueError('No observations at the requested cadence.')
    data = pd.concat(chunks).sort_index()
    data.index.name = 'decision_time'
    if not data.index.is_unique:
        raise ValueError('Duplicate decision_time keys. Fix the join upstream; do not drop duplicates blindly.')
    audit = {'raw_rows': raw_rows, 'sampled_rows': len(data), 'candidate_columns': len(values),
             'cadence_minutes': cadence, 'source_columns': len(names)}
    return data, values, audit

# Read-only identity for reproducibility; hashing a large Drive file may take time.
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

source_stat = Path(file_path).stat()
SOURCE_SHA256 = sha256_file(file_path)
data, VALUE_COLUMNS, load_audit = load_masked_dataset(file_path, TARGET, CADENCE_MINUTES)
print(load_audit)
print('Date range:', data.index.min(), 'to', data.index.max())
if 'label_status' in data:
    display(data['label_status'].value_counts(dropna=False).rename('rows').to_frame())
else:
    print('No label_status column: target validity can only be checked numerically here.')
display(data[VALUE_COLUMNS].isna().mean().describe().to_frame('missing_fraction'))

## 3. Admit labels and define the three evaluation months
If prompted, enter the exact status marking usable completed labels in your source, such as `VALID` **only if that is what your dataset uses**. Multiple accepted statuses can be comma-separated. Do not accept an incomplete/invalid status. No target is filled or converted to its absolute value automatically.

If a source month contains no usable labels, this code stops instead of silently choosing a different month. Partial final months are clearly labelled in the split table.

In [ ]:
if 'label_status' in data:
    observed = set(data.label_status.dropna().astype(str).unique())
    if VALID_LABEL_STATUSES is None:
        answer = input('Exact accepted completed label_status value(s), comma separated: ').strip()
        VALID_LABEL_STATUSES = [s.strip() for s in answer.split(',') if s.strip()]
    if not VALID_LABEL_STATUSES or not set(VALID_LABEL_STATUSES).issubset(observed):
        raise ValueError(f'Choose exact observed status values: {sorted(observed)}')
    status_ok = data.label_status.astype('string').isin(VALID_LABEL_STATUSES)
else:
    status_ok = pd.Series(True, index=data.index)
finite_y = pd.Series(np.isfinite(data[TARGET].to_numpy()), index=data.index)
if (data.loc[status_ok & finite_y, TARGET] < 0).any():
    raise ValueError('Negative net_magnitude_log_bps: check target semantics; do not silently apply abs().')
valid_y = status_ok & finite_y
print('Excluded missing/invalid labels:', int((~valid_y).sum()))
X = data.loc[valid_y, VALUE_COLUMNS].astype(np.float32)
y = data.loc[valid_y, TARGET].astype(float).copy()
label_end = pd.Series(X.index + HORIZON, index=X.index, name='label_end')
assert not set(DROP_FROM_X).intersection(X.columns)
assert all(c.lower().startswith('f4z__') and c.endswith('__value') for c in X.columns)
# The original DataFrame and Parquet are unchanged.
del data
print(f'X RAM: {X.memory_usage(deep=True).sum()/1024**2:.1f} MB; dtype: float32')

last_month = y.index.max().normalize().replace(day=1)
EVAL_START = last_month - pd.DateOffset(months=2)
EVAL_END = last_month + pd.DateOffset(months=1)
EVAL_MONTHS = pd.date_range(EVAL_START, EVAL_END, freq='MS', inclusive='left')
INNER_MONTHS = pd.date_range(EVAL_START - pd.DateOffset(months=3), EVAL_START, freq='MS', inclusive='left')

def split_indices(start, end, training_ceiling=None):
    train_mask = label_end < start  # strict endpoint purge, not a fixed row gap
    valid_mask = (X.index >= start) & (X.index < end)
    if training_ceiling is not None:
        # Last inner-validation labels must also mature before final evaluation.
        valid_mask &= (label_end < training_ceiling).to_numpy()
    tr, va = X.index[train_mask], X.index[valid_mask]
    if len(tr) < MIN_VALID or len(va) < 100:
        raise ValueError(f'Insufficient history at {start}: train={len(tr)}, valid={len(va)}')
    assert label_end.loc[tr].max() < va.min()
    return tr, va

INNER_SPLITS = [split_indices(s, s + pd.DateOffset(months=1), EVAL_START) for s in INNER_MONTHS]
OUTER_SPLITS = [split_indices(s, s + pd.DateOffset(months=1)) for s in EVAL_MONTHS]
split_rows = []
for kind, starts, splits in [('inner_tuning', INNER_MONTHS, INNER_SPLITS), ('outer_evaluation', EVAL_MONTHS, OUTER_SPLITS)]:
    for start, (tr, va) in zip(starts, splits):
        split_rows.append({'kind': kind, 'month': str(start.date()), 'train_rows': len(tr),
                           'validation_rows': len(va), 'last_train_label_end': label_end.loc[tr].max(),
                           'first_validation': va.min(), 'last_validation': va.max()})
split_table = pd.DataFrame(split_rows)
display(split_table)
split_table.to_csv(OUTPUT_ROOT / 'splits.csv', index=False)
print('Evaluation ends at last available labelled origin:', y.index.max())

## 4. Feature selection helpers
**Standalone MAE screen:** within each training fold, reserve its last calendar month for screening, purge 48-hour labels, fit one unpenalized median regression per admitted feature using earlier rows, and rank its screening MAE gain over the earlier-training median. Include a missing indicator and train-only median/scaling, as in the ZIP. Keep up to 20 positive-gain features; fall back to KS/lift if none improve.

**KS/lift alternative:** define a large move as magnitude above the earlier-screen-training 75th percentile. KS measures feature-distribution separation between these large moves and other moves on earlier-screen-training rows. Lift is the large-move rate of a feature quintile divided by the overall rate. Pick that quintile on earlier-screen-training only, then measure its lift in the screening validation month. Rank by combined KS and validated-lift ranks. These are screening heuristics; KS p-values are not used, and the regression target remains continuous magnitude.

The ZIP's original KS work used a different extreme-event definition. This quartile-based adaptation is explicit. The full-feature strategy remains eligible because weak standalone features can be useful jointly. No test-month labels select features or hyperparameters.

In [ ]:
FIT_LEDGER = []

def record_fit(kind, context, **fields):
    row = {'kind': kind, 'context': context, 'started_utc': datetime.now(timezone.utc).isoformat(),
           'status': 'started', **fields}
    FIT_LEDGER.append(row)
    return row

def admission(frame):
    finite = frame.replace([np.inf, -np.inf], np.nan)
    a = pd.DataFrame({'valid_count': finite.notna().sum(), 'unique_count': finite.nunique(dropna=True)})
    a['admitted'] = (a.valid_count >= MIN_VALID) & (a.unique_count >= 2)
    return a

def small_design(train_series, other_series):
    med = float(train_series.median())
    train_v = train_series.fillna(med).to_numpy(float)
    center, scale = train_v.mean(), train_v.std()
    scale = float(scale) if scale > 0 else 1.0
    def transform(s):
        return np.column_stack([(s.fillna(med).to_numpy(float) - center) / scale, s.isna().to_numpy(float)])
    return transform(train_series), transform(other_series)

def ks_lift(a, b, ya, yb):
    threshold = float(ya.quantile(0.75))
    event_a, event_b = ya.to_numpy() > threshold, yb.to_numpy() > threshold
    av, bv = a.to_numpy(float), b.to_numpy(float)
    present = np.isfinite(av)
    high, low = av[present & event_a], av[present & ~event_a]
    if min(len(high), len(low)) < 30:
        return 0.0, 1.0, 0
    ks = float(ks_2samp(high, low).statistic)
    edges = np.unique(np.quantile(av[present], [0.2, 0.4, 0.6, 0.8]))
    ba, bb = np.digitize(av, edges), np.digitize(bv, edges)
    base_a, base_b = float(event_a.mean()), float(event_b.mean())
    candidates = []
    for i in range(len(edges) + 1):
        m = present & (ba == i)
        if m.sum() >= 100 and base_a > 0:
            candidates.append((float(event_a[m].mean() / base_a), i))
    if not candidates:
        return ks, 1.0, 0
    _, chosen = max(candidates)
    m = np.isfinite(bv) & (bb == chosen)
    support = int(m.sum())
    validated_lift = float(event_b[m].mean() / base_b) if support >= 30 and base_b > 0 else 1.0
    return ks, validated_lift, support

def build_feature_sets(Xtrain, ytrain, context):
    admit = admission(Xtrain)
    all_cols = admit.index[admit.admitted].tolist()
    if not all_cols:
        raise ValueError('No admitted features; inspect masks/history.')
    # Dedicated chronological screening split nested inside this training frame.
    boundary = Xtrain.index.max() - pd.DateOffset(months=1)
    earlier = (Xtrain.index + HORIZON) < boundary
    later = Xtrain.index >= boundary
    a, b = Xtrain.loc[earlier], Xtrain.loc[later]
    ya, yb = ytrain.loc[a.index], ytrain.loc[b.index]
    if len(a) < MIN_VALID or len(b) < 100:
        raise ValueError('Insufficient data for nested feature screening.')
    assert (a.index + HORIZON).max() < b.index.min()
    screen_admit = admission(a)
    screen_cols = [c for c in all_cols if screen_admit.loc[c, 'admitted']]
    if not screen_cols:
        raise ValueError('No features admitted in the earlier screening segment.')
    baseline = mean_absolute_error(yb, np.full(len(yb), ya.median()))
    rows = []
    for i, col in enumerate(screen_cols):
        da, db = small_design(a[col], b[col])
        entry = record_fit('standalone_median', context, feature=col)
        try:
            reg = QuantileRegressor(quantile=0.5, alpha=0, solver='highs')
            reg.fit(da, ya)
            p = np.maximum(reg.predict(db), 0)
            mae = float(mean_absolute_error(yb, p))
            entry.update(status='success', screen_mae=mae)
        except Exception as exc:
            entry.update(status='failed', error=repr(exc))
            mae = np.nan
        ks, lift, support = ks_lift(a[col], b[col], ya, yb)
        rows.append({'feature': col, 'single_mae': mae, 'baseline_mae': baseline,
                     'mae_gain': baseline - mae, 'ks_statistic': ks,
                     'validated_tail_lift': lift, 'lift_validation_support': support})
        if (i + 1) % 50 == 0:
            print(f'{context}: screened {i+1}/{len(screen_cols)}', flush=True)
    ranking = pd.DataFrame(rows).set_index('feature')
    ranking['ks_lift_rank_score'] = (ranking.ks_statistic.rank(pct=True) +
                                    ranking.validated_tail_lift.rank(pct=True)) / 2
    ks_cols = ranking.sort_values(['ks_lift_rank_score', 'ks_statistic'], ascending=False).head(TOP_K).index.tolist()
    winners = ranking.loc[ranking.mae_gain > 0].sort_values('mae_gain', ascending=False).head(TOP_K).index.tolist()
    sets = {'all_admitted': all_cols, 'single_mae': winners or ks_cols, 'ks_lift': ks_cols}
    ranking['mae_screen_fell_back_to_ks_lift'] = not bool(winners)
    return sets, admit, ranking

def make_forest(params):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
        ('forest', RandomForestRegressor(criterion='absolute_error', bootstrap=True,
                                        random_state=SEED, n_jobs=RF_N_JOBS, **params))
    ])

## 5. Initialize MLflow and save the experiment contract
MLflow uses a local SQLite database with SQLite backup snapshots to Drive after each trial/fold. Artifacts are written to this experiment's Drive folder. This avoids an actively written SQLite database on a mounted Drive filesystem. Each run has its own database; it is not a shared multi-user MLflow server.

I/O writes target only the new output folder; no `to_parquet` or write call targets the original file. The final `mlflow.db`, predictions, exact environment, input hash, selected columns, screening tables, fit ledger, and monthly fitted pipelines will be saved. Failed screen fits remain in the ledger.

In [ ]:
TRACKING_DB = LOCAL_ROOT / 'mlflow.db'
TRACKING_URI = 'sqlite:///' + str(TRACKING_DB)
mlflow.set_tracking_uri(TRACKING_URI)
EXPERIMENT_NAME = 'BTC48_RF_' + RUN_ID
experiment_id = mlflow.create_experiment(EXPERIMENT_NAME, artifact_location=(OUTPUT_ROOT / 'mlflow_artifacts').as_uri())
mlflow.set_experiment(EXPERIMENT_NAME)

contract = dict(target=TARGET, horizon_hours=48, cadence_minutes=CADENCE_MINUTES,
                primary_metric='MAE_log_bps', min_valid=MIN_VALID, top_k=TOP_K,
                evaluation_start=str(EVAL_START), evaluation_end_exclusive=str(EVAL_END),
                evaluation_last_available=str(y.index.max()), configs_per_strategy=CONFIGS_PER_STRATEGY,
                seed=SEED, label_statuses=VALID_LABEL_STATUSES, source_sha256=SOURCE_SHA256,
                source_path=file_path, source_bytes=source_stat.st_size,
                strategy_selection='earlier purged inner CV only',
                target_definition_requires_source_confirmation=True,
                feature_screen='purged chronological standalone median; KS and validated tail lift fallback',
                rf_criterion='absolute_error', blend_weights=BLEND_WEIGHTS, memory_budget_mb=MAX_X_RAM_MB, label_purge='label_end strictly before validation start')
(OUTPUT_ROOT / 'contract.json').write_text(json.dumps(contract, indent=2))
(OUTPUT_ROOT / 'requirements_lock.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
(OUTPUT_ROOT / 'load_audit.json').write_text(json.dumps(load_audit, indent=2))

# Write snapshots safely; copy a completed backup rather than a live SQLite file.
def checkpoint():
    pd.DataFrame(FIT_LEDGER).to_csv(OUTPUT_ROOT / 'fit_ledger.csv', index=False)
    if TRACKING_DB.exists():
        backup = LOCAL_ROOT / 'mlflow_snapshot.db'
        with sqlite3.connect(TRACKING_DB) as src, sqlite3.connect(backup) as dst:
            src.backup(dst)
        shutil.copy2(backup, OUTPUT_ROOT / 'mlflow.db')

print('Tracking:', TRACKING_URI)
print('Expected RF fits:', 3 * CONFIGS_PER_STRATEGY * len(INNER_SPLITS) + len(OUTER_SPLITS))

## 6. Precompute training-only feature screens for the tuning folds
This can take time because standalone MAE regression fits are real models. All three RF strategies reuse the same fold-specific screens. Each screen is rebuilt from that fold's training rows; no global ranking is fitted before cross-validation.

In [ ]:
FOLD_CACHE = []
for fold, (tr, va) in enumerate(INNER_SPLITS):
    print('Preparing inner fold', fold + 1, flush=True)
    try:
        sets, admitted, ranks = build_feature_sets(X.loc[tr], y.loc[tr], f'inner_{fold+1}')
        admitted.to_csv(OUTPUT_ROOT / f'inner_{fold+1}_admission.csv')
        ranks.to_csv(OUTPUT_ROOT / f'inner_{fold+1}_feature_ranking.csv')
        FOLD_CACHE.append({'tr': tr, 'va': va, 'sets': sets})
    finally:
        checkpoint()
print('Feature counts:', [{k: len(v) for k, v in f['sets'].items()} for f in FOLD_CACHE])

## 7. Hyperparameter tuning — optimize pooled inner-fold MAE
Randomly sample bounded forest settings and evaluate each with every selection strategy. The same settings are used across strategies for a fair comparison. For every trial, also score `weight * RF + (1-weight) * training_median` with weights 0, 0.25, 0.5, 0.75, and 1 on inner folds; freeze the best weight. This adds no forest fits. Weight zero means RF failed to earn selection, not that it beat the baseline. Pure RF performance is reported separately. Score by total absolute error divided by total validation rows, rather than an unweighted average of unequal fold scores.

`criterion='absolute_error'` uses median leaves, but a forest averages its trees; this is not a guarantee of an exact conditional median estimator. MAE remains the selection/evaluation objective. No OOB or random train/test split is used as time-series validation.

In [ ]:
param_space = {
    'n_estimators': [100, 150],
    'max_depth': [4, 6, 8, 10],
    'min_samples_leaf': [50, 100, 200],
    'max_features': [0.25, 0.5, 1.0],
    'max_samples': [0.5, 0.8],
}
param_candidates = list(ParameterSampler(param_space, n_iter=CONFIGS_PER_STRATEGY, random_state=SEED))
# Convert NumPy scalars before JSON/MLflow logging.
param_candidates = [{k: (v.item() if isinstance(v, np.generic) else v) for k, v in p.items()} for p in param_candidates]
(OUTPUT_ROOT / 'search_candidates.json').write_text(json.dumps(param_candidates, indent=2))
trial_rows = []
for strategy in ['all_admitted', 'single_mae', 'ks_lift']:
    for candidate_id, params in enumerate(param_candidates):
        name = f'{strategy}_candidate_{candidate_id:02d}'
        print('Tuning:', name, params, flush=True)
        row = {'strategy': strategy, 'candidate_id': candidate_id, 'params': params, 'status': 'failed'}
        try:
            with mlflow.start_run(run_name=name):
                mlflow.log_params({'strategy': strategy, 'target': TARGET, 'criterion': 'absolute_error', **params})
                abs_sum = base_sum = nrows = 0
                blend_loss_sums = {w: 0.0 for w in BLEND_WEIGHTS}
                for fold, item in enumerate(FOLD_CACHE):
                    tr, va, cols = item['tr'], item['va'], item['sets'][strategy]
                    attempt = record_fit('random_forest', name, fold=fold+1, feature_count=len(cols))
                    try:
                        pipeline = make_forest(params)
                        pipeline.fit(X.loc[tr, cols], y.loc[tr])
                        prediction = np.maximum(pipeline.predict(X.loc[va, cols]), 0)
                        loss = np.abs(y.loc[va].to_numpy() - prediction)
                        baseline_value = float(y.loc[tr].median())
                        baseline_loss = np.abs(y.loc[va].to_numpy() - baseline_value)
                        for w in BLEND_WEIGHTS:
                            blend = w * prediction + (1-w) * baseline_value
                            blend_loss_sums[w] += float(np.abs(y.loc[va].to_numpy() - blend).sum())
                        abs_sum += loss.sum(); base_sum += baseline_loss.sum(); nrows += len(va)
                        attempt.update(status='success', validation_mae=float(loss.mean()))
                        mlflow.log_metric('fold_mae', float(loss.mean()), step=fold+1)
                        mlflow.log_metric('fold_baseline_mae', float(baseline_loss.mean()), step=fold+1)
                    except Exception as exc:
                        attempt.update(status='failed', error=repr(exc))
                        raise
                best_weight = min(BLEND_WEIGHTS, key=lambda w: blend_loss_sums[w])
                row.update(status='success', cv_mae=blend_loss_sums[best_weight]/nrows,
                           raw_rf_cv_mae=float(abs_sum/nrows), baseline_cv_mae=float(base_sum/nrows), blend_weight=best_weight)
                mlflow.log_param('selected_blend_weight', best_weight)
                mlflow.log_metrics({'cv_mae': row['cv_mae'], 'raw_rf_cv_mae': row['raw_rf_cv_mae'],
                                    'baseline_cv_mae': row['baseline_cv_mae']})
                for w in BLEND_WEIGHTS:
                    mlflow.log_metric('cv_mae_weight_' + str(w).replace('.', '_'), blend_loss_sums[w]/nrows)
                mlflow.log_artifact(str(OUTPUT_ROOT / 'contract.json'))
        except Exception as exc:
            row['error'] = repr(exc)
            print('Failed trial retained:', exc)
        finally:
            trial_rows.append(row)
            pd.DataFrame(trial_rows).to_json(OUTPUT_ROOT / 'tuning_trials.json', orient='records', indent=2)
            checkpoint()

tuning = pd.DataFrame(trial_rows)
success = tuning.loc[tuning.status == 'success'].sort_values('cv_mae')
if success.empty:
    raise RuntimeError('All tuning trials failed. Inspect fit_ledger.csv.')
best = success.iloc[0]
BEST_STRATEGY, BEST_PARAMS = best.strategy, best.params
BEST_WEIGHT = float(best.blend_weight)
selection = {'strategy': BEST_STRATEGY, 'params': BEST_PARAMS, 'blend_weight': BEST_WEIGHT, 'inner_cv_mae': float(best.cv_mae),
             'baseline_inner_cv_mae': float(best.baseline_cv_mae), 'frozen_before_outer_evaluation': True}
(OUTPUT_ROOT / 'frozen_selection.json').write_text(json.dumps(selection, indent=2))
display(success[['strategy', 'candidate_id', 'cv_mae', 'raw_rf_cv_mae', 'baseline_cv_mae', 'blend_weight', 'params']])
print('Frozen selection:', selection)
if best.cv_mae >= best.baseline_cv_mae:
    print('No RF/blend beat the median baseline in tuning. A zero-weight blend is the baseline, NOT a win. Pure RF is still reported for comparison.')

## 8. Three-month expanding walk-forward evaluation
The selected strategy and hyperparameters stay fixed. Feature membership, imputation, and the forest are refitted using each month's admissible history. That distinction makes this an evaluation of a fixed **training procedure**, not one static fitted model. No monthly scores change the procedure. The blending weight is also frozen from earlier folds; the median component is recomputed using only the current training history.

Models and preprocessing are saved together. To replay a month, load its joblib pipeline and select exactly the columns in its adjacent JSON file after applying the same validity/availability masks. The saved pipeline outputs pure RF; apply `blend.json` as `rf_weight * raw_rf + (1-rf_weight) * baseline_value` to reproduce the selected forecast.

In [ ]:
outer_predictions, monthly_rows = [], []
for fold, (month, (tr, va)) in enumerate(zip(EVAL_MONTHS, OUTER_SPLITS), start=1):
    tag = month.strftime('%Y-%m')
    folder = OUTPUT_ROOT / f'walk_forward_{tag}'
    folder.mkdir(exist_ok=True)
    print('Walk-forward:', tag, 'training rows:', len(tr), 'test rows:', len(va), flush=True)
    try:
        if BEST_STRATEGY == 'all_admitted':
            admitted = admission(X.loc[tr])
            cols = admitted.index[admitted.admitted].tolist()
            ranks = pd.DataFrame()
        else:
            sets, admitted, ranks = build_feature_sets(X.loc[tr], y.loc[tr], f'outer_{tag}')
            cols = sets[BEST_STRATEGY]
        if not cols:
            raise ValueError('No admitted features for evaluation.')
        admitted.to_csv(folder / 'admission.csv')
        ranks.to_csv(folder / 'feature_ranking.csv')
        (folder / 'selected_features.json').write_text(json.dumps(cols, indent=2))
        with mlflow.start_run(run_name=f'walk_forward_{tag}'):
            mlflow.log_params({'phase': 'outer_evaluation', 'month': tag, 'strategy': BEST_STRATEGY,
                               'target': TARGET, 'n_features': len(cols), 'criterion': 'absolute_error', 'blend_weight': BEST_WEIGHT, **BEST_PARAMS})
            mlflow.set_tags({'last_train_label_end': str(label_end.loc[tr].max()),
                             'first_test_origin': str(va.min()), 'source_sha256': SOURCE_SHA256})
            attempt = record_fit('random_forest', f'outer_{tag}', feature_count=len(cols))
            try:
                fitted = make_forest(BEST_PARAMS)
                fitted.fit(X.loc[tr, cols], y.loc[tr])
                raw_rf_prediction = np.maximum(fitted.predict(X.loc[va, cols]), 0)
                baseline_value = float(y.loc[tr].median())
                predicted = BEST_WEIGHT * raw_rf_prediction + (1-BEST_WEIGHT) * baseline_value
                if not np.isfinite(predicted).all():
                    raise ValueError('Nonfinite forecast.')
                attempt['status'] = 'success'
            except Exception as exc:
                attempt.update(status='failed', error=repr(exc))
                raise
            p = pd.DataFrame({'actual': y.loc[va], 'prediction': predicted, 'raw_rf_prediction': raw_rf_prediction,
                              'median_baseline': float(y.loc[tr].median()), 'fold': tag}, index=va)
            p['absolute_error'] = np.abs(p.actual - p.prediction)
            p['raw_rf_absolute_error'] = np.abs(p.actual - p.raw_rf_prediction)
            p['baseline_absolute_error'] = np.abs(p.actual - p.median_baseline)
            p['paired_loss_difference'] = p.absolute_error - p.baseline_absolute_error
            metrics = {'mae': float(p.absolute_error.mean()), 'baseline_mae': float(p.baseline_absolute_error.mean()),
                       'delta_mae': float(p.paired_loss_difference.mean()), 'raw_rf_mae': float(p.raw_rf_absolute_error.mean())}
            mlflow.log_metrics(metrics)
            p.to_csv(folder / 'predictions.csv', index_label='decision_time')
            joblib.dump(fitted, folder / 'pipeline.joblib', compress=3)
            (folder / 'blend.json').write_text(json.dumps({'rf_weight': BEST_WEIGHT, 'baseline_value': baseline_value}, indent=2))
            # Standard pipeline is logged as a model; columns/masks are separate explicit artifacts.
            mlflow.sklearn.log_model(fitted, artifact_path='model', input_example=X.loc[tr, cols].head(3))
            mlflow.log_artifacts(str(folder), artifact_path='audit_and_replay')
            outer_predictions.append(p)
            monthly_rows.append({'month': tag, 'rows': len(va), 'features': len(cols), **metrics})
    finally:
        checkpoint()

predictions = pd.concat(outer_predictions).sort_index()
expected_keys = X.index[(X.index >= EVAL_START) & (X.index < EVAL_END)]
assert predictions.index.equals(expected_keys), 'Missing or extra evaluation rows.'
assert predictions.index.is_unique
monthly = pd.DataFrame(monthly_rows)
pooled_mae = float(predictions.absolute_error.mean())
pooled_baseline_mae = float(predictions.baseline_absolute_error.mean())
relative_gain = 100 * (1 - pooled_mae/pooled_baseline_mae) if pooled_baseline_mae > 0 else np.nan
print(f'Pooled selected forecast MAE: {pooled_mae:.6f} log bps; RF weight={BEST_WEIGHT}')
print(f'Pooled pure RF MAE: {predictions.raw_rf_absolute_error.mean():.6f} log bps')
print(f'Pooled training-median baseline MAE: {pooled_baseline_mae:.6f} log bps')
print(f'MAE improvement vs baseline: {relative_gain:.3f}%')
display(monthly)
predictions.to_csv(OUTPUT_ROOT / 'walk_forward_predictions.csv', index_label='decision_time')
monthly.to_csv(OUTPUT_ROOT / 'monthly_metrics.csv', index=False)

## 9. Paired block uncertainty and diagnostics
Negative loss differences favor the forest. The bootstrap resamples paired calendar blocks, preserving the forest and baseline losses together. These 95% intervals are nominal diagnostics, not a correction for all model searches or prior inspection of this dataset. Three months provide relatively few 14-day blocks.

Monthly and pooled metrics describe a single frozen walk-forward procedure. Do not choose another strategy based on these final results and continue calling the same period an untouched test.

In [ ]:
def paired_block_intervals(p, draws=2000):
    results = []
    delta = p['paired_loss_difference']
    for days in [7, 14]:
        groups = ((delta.index - delta.index.min()).total_seconds() // (days * 86400)).astype(int)
        sums = delta.groupby(groups).sum().to_numpy()
        sizes = delta.groupby(groups).size().to_numpy()
        rng = np.random.default_rng(SEED + days)
        selected = rng.integers(0, len(sizes), size=(draws, len(sizes)))
        sampled = sums[selected].sum(axis=1) / sizes[selected].sum(axis=1)
        lo, hi = np.quantile(sampled, [0.025, 0.975])
        results.append({'block_days': days, 'blocks': len(sizes), 'mean_delta_mae': float(delta.mean()),
                        'ci_95_low': float(lo), 'ci_95_high': float(hi), 'interpretation': 'negative favors RF; nominal only'})
    return pd.DataFrame(results)

intervals = paired_block_intervals(predictions, N_BOOTSTRAP)
display(intervals)
intervals.to_csv(OUTPUT_ROOT / 'paired_block_intervals.csv', index=False)
fig, axes = plt.subplots(2, 1, figsize=(12, 8), constrained_layout=True)
monthly.set_index('month')[['mae', 'raw_rf_mae', 'baseline_mae']].plot.bar(ax=axes[0], rot=0)
axes[0].set_ylabel('MAE (log bps)'); axes[0].set_title('Monthly walk-forward MAE — lower is better')
predictions.paired_loss_difference.cumsum().plot(ax=axes[1])
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_ylabel('Cumulative paired absolute-loss difference')
axes[1].set_title('Below zero favors the frozen selected forecast over the training-median baseline')
fig.savefig(OUTPUT_ROOT / 'evaluation.png', dpi=160)
plt.show()

summary = {'selected_forecast_mae_log_bps': pooled_mae,
           'raw_rf_mae_log_bps': float(predictions.raw_rf_absolute_error.mean()), 'rf_blend_weight': BEST_WEIGHT, 'median_baseline_mae_log_bps': pooled_baseline_mae,
           'delta_mae': pooled_mae-pooled_baseline_mae,
           'improvement_percent': float(relative_gain) if np.isfinite(relative_gain) else None,
           'evaluation_rows': len(predictions), 'months_better': int((monthly.mae < monthly.baseline_mae).sum()),
           'beats_baseline': bool(BEST_WEIGHT > 0 and pooled_mae < pooled_baseline_mae - 1e-12),
           'pure_rf_beats_baseline': bool(predictions.raw_rf_absolute_error.mean() < pooled_baseline_mae - 1e-12),
           'status': 'walk_forward_research_result_not_confirmed_alpha'}
(OUTPUT_ROOT / 'summary.json').write_text(json.dumps(summary, indent=2))
print('Selected forecast beats baseline:', summary['beats_baseline'])
print('Pure Random Forest beats baseline:', summary['pure_rf_beats_baseline'])
with mlflow.start_run(run_name='pooled_walk_forward_summary'):
    mlflow.log_params({'phase': 'summary', 'strategy': BEST_STRATEGY, 'target': TARGET})
    mlflow.log_metrics({k: float(v) for k, v in summary.items() if isinstance(v, (float, int))})
    for name in ['summary.json', 'monthly_metrics.csv', 'paired_block_intervals.csv', 'walk_forward_predictions.csv',
                 'evaluation.png', 'frozen_selection.json', 'requirements_lock.txt', 'contract.json']:
        mlflow.log_artifact(str(OUTPUT_ROOT / name))
checkpoint()
print(pd.DataFrame(FIT_LEDGER).groupby(['kind', 'status']).size())
# No code writes to file_path; this additionally detects size/mtime changes during the run.
end_stat = Path(file_path).stat()
assert (source_stat.st_size, source_stat.st_mtime_ns) == (end_stat.st_size, end_stat.st_mtime_ns), 'Source changed during the run.'
print('Completed. All outputs:', OUTPUT_ROOT)

## 10. View MLflow runs in the notebook
The table below works without a public tunnel. A local UI can also be opened through Colab's proxy; if your browser blocks it, use the table. This is a local experiment browser, not a team-accessible server.

To restore after a Colab reset, mount Drive, copy the saved `mlflow.db` to a local `/content/...` directory, set `mlflow.set_tracking_uri('sqlite:////content/.../mlflow.db')`, and use the saved experiment name. Drive must be mounted at the same path for existing artifact URIs to resolve.

In [ ]:
runs = mlflow.search_runs(experiment_ids=[experiment_id], order_by=['start_time DESC'])
show_cols = [c for c in ['run_id', 'tags.mlflow.runName', 'status', 'params.strategy',
                        'metrics.cv_mae', 'metrics.mae', 'metrics.baseline_mae', 'metrics.selected_forecast_mae_log_bps'] if c in runs]
display(runs[show_cols])
runs.to_csv(OUTPUT_ROOT / 'mlflow_runs.csv', index=False)
print('Saved tracking database:', OUTPUT_ROOT / 'mlflow.db')

In [ ]:
# Optional UI: run this cell only if you want the MLflow browser.
# Keep this within your Colab session; do not expose an unauthenticated public tunnel.
OPEN_MLFLOW_UI = False
if OPEN_MLFLOW_UI:
    ui_log = open(LOCAL_ROOT / 'mlflow_ui.log', 'w')
    ui_process = subprocess.Popen([
        sys.executable, '-m', 'mlflow', 'ui', '--backend-store-uri', TRACKING_URI,
        '--host', '0.0.0.0', '--port', '5000'
    ], stdout=ui_log, stderr=subprocess.STDOUT)
    from google.colab import output
    output.serve_kernel_port_as_window(5000)

## Reading the outputs
- `frozen_selection.json`: winning training-only feature strategy and hyperparameters.
- `inner_*_feature_ranking.csv`: standalone MAE, KS, and validated lift scores.
- `walk_forward_YYYY-MM/selected_features.json`: exact X columns used that month.
- `walk_forward_YYYY-MM/pipeline.joblib`: imputer and forest for reproducible prediction.
- `monthly_metrics.csv`, `summary.json`: monthly and pooled MAE.
- `walk_forward_predictions.csv`: every scored timestamp, truth, RF prediction, baseline, paired loss.
- `paired_block_intervals.csv`: nominal 7/14-day paired uncertainty.
- `fit_ledger.csv`: attempted standalone and forest fits, including failures.
- `mlflow.db`, `mlflow_artifacts/`, `mlflow_runs.csv`: MLflow tracking records and artifacts.

**Memory safeguards:** batches of 4,096 source rows, float32 feature storage, a 750 MB X-size guard, sequential trials, two forest workers, bounded depths, no full feature correlation matrix, and no full-data SHAP. Total peak RAM exceeds X size because fitting requires working arrays; lower `RF_N_JOBS` to 1 if necessary. The source file is never written.

**What this cannot establish:** the supplied column names do not reveal actual usable coverage or predictive value; those are computed when you run the notebook. The validity/availability fields cannot repair upstream leakage. The last three months may already have been inspected historically. This code does not train an expected signed-return model or demonstrate executable trading profit.

## Documentation
- [RandomForestRegressor: absolute-error criterion](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)
- [QuantileRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.QuantileRegressor.html)
- [MLflow local SQLite tracking](https://mlflow.org/docs/latest/ml/tracking/tutorials/local-database/)

The pipeline follows the provided ZIP's training-only masks/admission, missingness handling, chronological purging, magnitude MAE, and paired uncertainty principles. Its dates, RF search, and KS/lift screen are explicitly new choices.